# Câu 1: Nhập dữ liệu và thống kê thông tin cơ bản của các thuộc tính

Phần 1 – Bài 2 (trang 28): Khảo sát và tiền xử lý dữ liệu Bank marketing giống phần hướng dẫn.

Dữ liệu dùng: `bank-full.csv` (45.211 khách hàng, 16 thuộc tính + thuộc tính quyết định `y` – có đăng ký gửi tiền có kỳ hạn hay không).

In [1]:
SV = "Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT"
DATA = "../../data/"  # thư mục chứa dữ liệu
print(SV)

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT


In [ ]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

---
## Thực hiện yêu cầu

In [3]:
print(SV)
df = pd.read_csv(DATA + 'bank-full.csv', sep=';')
df.head(10)

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
5,35,management,married,tertiary,no,231,yes,no,unknown,5,may,139,1,-1,0,unknown,no
6,28,management,single,tertiary,no,447,yes,yes,unknown,5,may,217,1,-1,0,unknown,no
7,42,entrepreneur,divorced,tertiary,yes,2,yes,no,unknown,5,may,380,1,-1,0,unknown,no
8,58,retired,married,primary,no,121,yes,no,unknown,5,may,50,1,-1,0,unknown,no
9,43,technician,single,secondary,no,593,yes,no,unknown,5,may,55,1,-1,0,unknown,no


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


In [5]:
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [6]:
df.select_dtypes(exclude='number').describe()

,job,marital,education,default,housing,loan,contact,month,poutcome,y
count,45211,45211,45211,45211,45211,45211,45211,45211,45211,45211
unique,12,3,4,2,2,2,3,12,4,2
top,blue-collar,married,secondary,no,yes,no,cellular,may,unknown,no
freq,9732,27214,23202,44396,25130,37967,29285,13766,36959,39922


**Phân loại thuộc tính:**

| Thuộc tính | Loại |
|---|---|
| job, marital, contact, poutcome | Nominal |
| education | Ordinal (primary < secondary < tertiary) |
| month | Ordinal (theo thứ tự tháng) |
| default, housing, loan | Binary (bất đối xứng) |
| y | Binary – thuộc tính quyết định |
| age, balance, duration, campaign, previous | Quantitative – ratio-scaled |
| day | Quantitative – interval-scaled |
| pdays | Quantitative (−1 = chưa từng liên hệ) |

**Nhận xét:**

- Dữ liệu `bank-full.csv` có 45.211 khách hàng, 16 thuộc tính đầu vào và thuộc tính quyết định `y`. Bộ này là phiên bản `bank` (thuộc tính `balance`, `poutcome`...) nên khác một chút so với mẫu trong đề (bản `bank-additional` có `cons.price.idx`).
- Dữ liệu mất cân bằng: chỉ 5.289/45.211 (≈11,7%) khách đăng ký gửi tiền có kỳ hạn.
- Tuổi trung bình ≈41 (18–95). `balance` lệch rất mạnh (−8.019 đến 102.127, trung vị 448); `duration`, `campaign`, `previous` cũng có ngoại lai lớn (ví dụ `previous` tối đa 275).
- Khách phổ biến nhất: nghề blue-collar, đã kết hôn, học vấn trung học (secondary), không nợ xấu.